In [1]:
import sys
import os
import pandas as pd
import numpy as np
import xgboost as xgb

from sklearn.metrics import ndcg_score

# 1つ上のディレクトリにパスを通す
sys.path.append(os.path.abspath('../'))

from module.Dataset import read_preprocess_tsv, read_test_tsv, train_division, ColumnEncode


In [2]:
# 前処理済みデータの読み込み
train_A = read_preprocess_tsv('A')


In [3]:
# 予測用データの読み込み
test_A = read_test_tsv('A')


In [4]:
# エンコードクラスのインスタンス化
user_item_enc = ColumnEncode()

In [5]:
# エンコードの実行
train_A['user_id'] = user_item_enc.user_encode(train_A['user_id'])
train_A['product_id'] = user_item_enc.product_encode(train_A['product_id'])

test_A['user_id'] = user_item_enc.test_encode(test_A['user_id'])

In [6]:
# タイムスタンプを数値型に変換
train_A["time_stamp"] = pd.to_datetime(train_A["time_stamp"])
train_A['unix_time'] = train_A['time_stamp'].astype(np.int64) // 10**9  # UNIX時間に変換


In [7]:
# 学習データの分割
X_train, y_train, X_valid, y_valid = train_division(train_A)


In [8]:
# 特徴量とターゲット変数の設定
features = [col for col in X_train.columns if col not in ['relevance', 'time_stamp', 'weekday', 'date']]

X_train_feat = X_train[features]
X_valid_feat = X_valid[features]
X_train_target = X_train['relevance']
X_valid_target = X_valid[['user_id', 'relevance']]


In [9]:
# 訓練データの特徴量名
train_features = X_train_feat.columns


In [10]:
# 訓練データ(X_train)の特徴量の順番にX_valid_featの順番を合わせる
X_valid_feat = X_valid_feat[train_features]


In [11]:
# モデル定義（固定ハイパーパラメータ）
model = xgb.XGBRegressor(
    objective='reg:squarederror',
    eval_metric='rmse',
    learning_rate=0.05,
    max_depth=6,
    n_estimators=200,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=1,
    gamma=0.3,
    reg_alpha=0.1,
    reg_lambda=1.0,
    tree_method='hist',
    grow_policy='lossguide'
)


In [13]:
# モデル学習
model.fit(X_train_feat, X_train_target)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric='rmse', feature_types=None,
             gamma=0.3, grow_policy='lossguide', importance_type=None,
             interaction_constraints=None, learning_rate=0.05, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=1, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=200, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

In [14]:
# 推薦関数
def recommend_products(user_id, data_source, top_k=22):
    user_data = data_source[data_source['user_id'] == user_id][['user_id', 'product_id', 'ad', 'unix_time']]
    if user_data.empty:
        return pd.DataFrame(columns=['user_id', 'product_id', 'rank'])
    predictions = model.predict(user_data, validate_features=False)
    user_data = user_data.copy()
    user_data['score'] = predictions
    recommendations = user_data.sort_values(by='score', ascending=False).head(top_k)
    recommendations['rank'] = range(1, len(recommendations) + 1)
    recommendations['user_id'] = user_item_enc.user_decode(recommendations['user_id'])
    recommendations['product_id'] = user_item_enc.product_decode(recommendations['product_id'])
    return recommendations[['user_id', 'product_id', 'rank']]


In [15]:
# nDCG評価
def evaluate_ndcg(y_true_data, candidate_data):
    y_true = []
    y_score = []
    for user_id in y_true_data['user_id'].unique():
        actual = y_true_data[y_true_data['user_id'] == user_id].sort_values(by='relevance', ascending=False)['relevance'].values
        pred_df = recommend_products(user_id, candidate_data, top_k=len(actual))
        pred = np.linspace(3, 1, len(pred_df))  # スコアを仮に降順とする
        if len(pred) == 0:
            continue
        y_true.append(actual)
        y_score.append(pred)
    return np.mean([ndcg_score([t], [s]) for t, s in zip(y_true, y_score)]) if y_true else 0.0


In [16]:
# 評価
print("Validation nDCG Score:", evaluate_ndcg(X_valid_target, X_valid_feat))


XGBoostError: [18:17:04] /workspace/src/predictor/cpu_predictor.cc:789: Check failed: m->NumColumns() == model.learner_model_param->num_feature (4 vs. 72) : Number of columns in data must equal to trained model.
Stack trace:
  [bt] (0) /home/mayrin/develop/venv/lib/python3.12/site-packages/xgboost/lib/libxgboost.so(+0x25c1ac) [0x7f5f9c45c1ac]
  [bt] (1) /home/mayrin/develop/venv/lib/python3.12/site-packages/xgboost/lib/libxgboost.so(+0x75ea80) [0x7f5f9c95ea80]
  [bt] (2) /home/mayrin/develop/venv/lib/python3.12/site-packages/xgboost/lib/libxgboost.so(+0x75f6f6) [0x7f5f9c95f6f6]
  [bt] (3) /home/mayrin/develop/venv/lib/python3.12/site-packages/xgboost/lib/libxgboost.so(+0x5aa1e8) [0x7f5f9c7aa1e8]
  [bt] (4) /home/mayrin/develop/venv/lib/python3.12/site-packages/xgboost/lib/libxgboost.so(+0x5fde69) [0x7f5f9c7fde69]
  [bt] (5) /home/mayrin/develop/venv/lib/python3.12/site-packages/xgboost/lib/libxgboost.so(+0x16a7ac) [0x7f5f9c36a7ac]
  [bt] (6) /home/mayrin/develop/venv/lib/python3.12/sit

In [ ]:
# 予測結果保存
def save_predictions():
    results = []
    for user_id in test_A['user_id'].unique():
        if user_id in test_A['user_id'].values:
            recommendations = recommend_products(user_id, test_A)
            for _, row in recommendations.iterrows():
                results.append([row['user_id'], row['product_id'], row['rank']])
    if not results:
        print("Warning: No predictions generated.")
    pd.DataFrame(results, columns=['user_id', 'product_id', 'rank']).to_csv('../dataset/predictions_A.tsv', sep='\t', index=False, header=False)


In [ ]:
save_predictions()
